In [1]:
# ============================================================================
# CalEdgeFormer — Step 10, Part 2 of 4: temperature scaling (self-contained)
# ============================================================================
# Purpose: post-hoc calibration on the sigmoid baseline from Part 1. This is
# NOT a training run — it loads Part 1's checkpoint, runs it once over val/
# to get logits, then fits a single scalar temperature T (Guo et al. 2017).
# Fast (a couple of minutes), no epochs.
#
# REQUIRES: step10_sigmoid_baseline_best.pt from Part 1.
#   - If you're running this in the SAME Kaggle session as Part 1, it's
#     already sitting in /kaggle/working/checkpoints/ — nothing to do.
#   - If you're running this in a FRESH session, add the checkpoint as an
#     input dataset first (Add Data -> Upload -> the .pt file you downloaded
#     after Part 1), then this script will find it under /kaggle/input.
#
# Paste each "CELL" block into its own Kaggle notebook cell, in order.
# Settings -> Accelerator -> GPU T4 x2 or GPU P100 (a GPU still helps for the
# one forward pass over val/, though this part is far lighter than training).
# ============================================================================


# ===== CELL 1: environment check =====
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ===== CELL 2: locate and verify the WHU dataset under /kaggle/input =====
import os

def find_whu_root(base='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, s)) for s in ('train', 'val', 'test')):
            if all(os.path.isdir(os.path.join(dirpath, s, 'Image')) and
                   os.path.isdir(os.path.join(dirpath, s, 'Mask'))
                   for s in ('train', 'val', 'test')):
                return dirpath
    return None

root = find_whu_root()
if root is None:
    # root = '/kaggle/input/whu-building-dataset/WHU'  # set manually if needed
    raise RuntimeError('Could not auto-detect the WHU dataset under /kaggle/input.')

print('Using dataset root:', root)
val_count = len(os.listdir(os.path.join(root, 'val', 'Image')))
print('val images:', val_count)

ckpt_dir = '/kaggle/working/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)


# ===== CELL 3: locate Part 1's sigmoid checkpoint =====
SIGMOID_CKPT_NAME = 'step10_sigmoid_baseline_best.pt'

def find_file(filename, search_roots=('/kaggle/working', '/kaggle/input')):
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None

sigmoid_ckpt_path = find_file(SIGMOID_CKPT_NAME)
if sigmoid_ckpt_path is None:
    raise RuntimeError(
        f'Could not find {SIGMOID_CKPT_NAME} under /kaggle/working or /kaggle/input. '
        'If this is a fresh session (not the same one Part 1 ran in), add it as an '
        'input dataset first (Add Data -> Upload -> the .pt file you downloaded '
        'after Part 1), then re-run this cell.'
    )
print('Found sigmoid checkpoint at:', sigmoid_ckpt_path)


# ===== CELL 4: class/function definitions needed to load the sigmoid model =====
import json
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class CrossScaleTransformerFusion(nn.Module):
    def __init__(self, dim=1024, num_heads=8, mlp_ratio=4):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        q = self.q_proj(tokens)
        k = self.k_proj(tokens)
        v = self.v_proj(tokens)
        N = tokens.shape[1]
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)
        tokens = self.norm1(tokens + out)
        mlp_out = self.mlp(tokens)
        tokens = self.norm2(tokens + mlp_out)
        out_spatial = tokens.transpose(1, 2).reshape(B, C, H, W)
        return out_spatial


class EdgeAwareAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.conv_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        gx = F.conv2d(x, self.sobel_x, padding=1, groups=self.channels)
        gy = F.conv2d(x, self.sobel_y, padding=1, groups=self.channels)
        edge = torch.sqrt(gx * gx + gy * gy + 1e-6)
        feat = self.conv_branch(x)
        combined = torch.cat([edge, feat], dim=1)
        gate_map = self.gate(combined)
        return x * gate_map


class EdgeFormerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResBlock(32, 32, stride=2)
        self.enc2 = ResBlock(32, 64, stride=2)
        self.enc3 = ResBlock(64, 128, stride=2)
        self.enc4 = ResBlock(128, 256, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 1024, 3, padding=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.cstf = CrossScaleTransformerFusion(dim=1024, num_heads=8, mlp_ratio=4)
        self.up1 = nn.ConvTranspose2d(1024, 256, 2, stride=2)
        self.dec1 = ResBlock(256 + 128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResBlock(128 + 64, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ResBlock(64 + 32, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec4 = ResBlock(32 + 32, 32)
        self.eaa = EdgeAwareAttention(32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        b = self.cstf(b)
        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1, e3], dim=1))
        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d3 = self.up3(d2)
        d3 = self.dec3(torch.cat([d3, e1], dim=1))
        d4 = self.up4(d3)
        d4 = self.dec4(torch.cat([d4, e0], dim=1))
        d4 = self.eaa(d4)
        out = self.head(d4)
        return out


# ---- dataset (val/ only needed here — no augmentation regardless) ----
MEAN = np.array([0.44231387226534, 0.44906677331805, 0.41436488550588], dtype=np.float32)
STD = np.array([0.21254212670769, 0.19898734700646, 0.21256595675766], dtype=np.float32)

class WHUDataset(Dataset):
    def __init__(self, root, split, train=False):
        self.root = root
        self.split = split
        self.train = train
        self.files = sorted(os.listdir(os.path.join(root, split, 'Image')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.root, self.split, 'Image', fname)).convert('RGB')
        mask = Image.open(os.path.join(self.root, self.split, 'Mask', fname)).convert('L')
        img_arr = np.asarray(img, dtype=np.float32) / 255.0
        img_arr = (img_arr - MEAN) / STD
        mask_arr = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)
        img_t = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_arr).unsqueeze(0).float()
        return img_t, mask_t


def fit_temperature_scaling(model, val_loader, device, ckpt_dir, sigmoid_ckpt_path):
    model.eval()
    logits_list = []
    labels_list = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            logits_list.append(logits.cpu())
            labels_list.append(yb)
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list)

    T = torch.nn.Parameter(torch.ones(1) * 1.5)
    optimizer = torch.optim.LBFGS([T], lr=0.05, max_iter=50)

    def closure():
        optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(logits / T.clamp(min=0.05), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    T_final = float(T.clamp(min=0.05).item())

    pre_nll = F.binary_cross_entropy_with_logits(logits, labels).item()
    post_nll = F.binary_cross_entropy_with_logits(logits / T_final, labels).item()

    os.makedirs(ckpt_dir, exist_ok=True)
    cfg_path = os.path.join(ckpt_dir, 'step10_temp_scaling_config.json')
    config = {
        'step': '10_baselines',
        'model_tag': 'temp_scaling',
        'base_model_checkpoint': sigmoid_ckpt_path,
        'method': 'single scalar temperature T fit via LBFGS minimizing NLL on val/ logits (Guo et al. 2017)',
        'temperature': T_final,
        'val_nll_before_scaling': pre_nll,
        'val_nll_after_scaling': post_nll,
    }
    with open(cfg_path, 'w') as f:
        json.dump(config, f, indent=2)
    print('temperature scaling: T=', T_final, 'val_nll before=', pre_nll, 'after=', post_nll)
    return T_final, config

print('Part 2 class/function definitions loaded OK.')


# ===== CELL 5: build val loader + load the sigmoid checkpoint =====
val_ds = WHUDataset(root, 'val', train=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
print('val batches:', len(val_loader))

model_for_ts = EdgeFormerNet().to(device)
model_for_ts.load_state_dict(torch.load(sigmoid_ckpt_path, map_location=device))
print('Loaded sigmoid checkpoint from', sigmoid_ckpt_path)


# ===== CELL 6: fit temperature scaling =====
T_final, cfg_temp = fit_temperature_scaling(model_for_ts, val_loader, device, ckpt_dir, sigmoid_ckpt_path)
print(json.dumps(cfg_temp, indent=2))


# ===== CELL 7: summary — copy this cell's output back to chat =====
print('=' * 70)
print('STEP 10 PART 2 SUMMARY (temperature scaling)')
print('=' * 70)
print('checkpoint dir contents (', ckpt_dir, '):')
for f in sorted(os.listdir(ckpt_dir)):
    print(' ', f)
print()
print(json.dumps(cfg_temp, indent=2))
print()
print('No .pt file for this part — temperature scaling only produces a config')
print('JSON (the scalar T value + a pointer to the sigmoid checkpoint it was')
print('fit on). Nothing new to download here beyond step10_temp_scaling_config.json')
print('if you want it as a standalone file; it is also already embedded above.')


CUDA available: True
Device: Tesla T4
Using dataset root: /kaggle/input/datasets/sengulgs/whu-building-dataset/WHU
val images: 1228
Found sigmoid checkpoint at: /kaggle/input/datasets/saravanachandranw/step10-sigmoid-baseline-pt/step10_sigmoid_baseline_best.pt
Part 2 class/function definitions loaded OK.
val batches: 154
Loaded sigmoid checkpoint from /kaggle/input/datasets/saravanachandranw/step10-sigmoid-baseline-pt/step10_sigmoid_baseline_best.pt
temperature scaling: T= 1.329512119293213 val_nll before= 0.10378910601139069 after= 0.09824657440185547
{
  "step": "10_baselines",
  "model_tag": "temp_scaling",
  "base_model_checkpoint": "/kaggle/input/datasets/saravanachandranw/step10-sigmoid-baseline-pt/step10_sigmoid_baseline_best.pt",
  "method": "single scalar temperature T fit via LBFGS minimizing NLL on val/ logits (Guo et al. 2017)",
  "temperature": 1.329512119293213,
  "val_nll_before_scaling": 0.10378910601139069,
  "val_nll_after_scaling": 0.09824657440185547
}
STEP 10 PART 

In [2]:
# ============================================================================
# CalEdgeFormer — Step 10, Part 1 of 4: sigmoid baseline (self-contained)
# ============================================================================
# Purpose: step 10 already PASSED (config JSON already recorded) — this run
# exists only to regenerate the real .pt weight file, since the original
# Kaggle session's checkpoints folder was lost before it could be downloaded.
#
# Paste each "CELL" block into its own Kaggle notebook cell, in order.
# Before running: Settings (right sidebar) -> Accelerator -> GPU T4 x2 or
# GPU P100. No internet access needed.
#
# This part is fully self-contained — no dependency on any other part.
# Part 2 (temperature scaling) will need THIS part's checkpoint file, so
# run this one first if you're doing Part 2 next.
#
# After it finishes: Save Version -> Save & Run All (Commit) -> check the
# Output tab for /kaggle/working/checkpoints/step10_sigmoid_baseline_best.pt
# and _last.pt -> download -> upload into Drive's Dataset WHU/checkpoints/.
# ============================================================================


# ===== CELL 1: environment check =====
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (stop and enable GPU accelerator!)')
assert torch.cuda.is_available(), 'No GPU — enable it in Settings before continuing.'
device = torch.device('cuda')


# ===== CELL 2: locate and verify the WHU dataset under /kaggle/input =====
import os

def find_whu_root(base='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, s)) for s in ('train', 'val', 'test')):
            if all(os.path.isdir(os.path.join(dirpath, s, 'Image')) and
                   os.path.isdir(os.path.join(dirpath, s, 'Mask'))
                   for s in ('train', 'val', 'test')):
                return dirpath
    return None

root = find_whu_root()
if root is None:
    # Auto-detect failed — set this manually, e.g.:
    # root = '/kaggle/input/whu-building-dataset/WHU'
    raise RuntimeError(
        'Could not auto-detect the WHU dataset under /kaggle/input. '
        'Set `root` manually to the folder containing train/val/test.'
    )

print('Using dataset root:', root)
for s in ['train', 'val', 'test']:
    imgs = os.listdir(os.path.join(root, s, 'Image'))
    msks = os.listdir(os.path.join(root, s, 'Mask'))
    print(s, 'images:', len(imgs), 'masks:', len(msks))

ckpt_dir = '/kaggle/working/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)


# ===== CELL 3: class/function definitions needed for the sigmoid baseline =====
import json
import math
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class CrossScaleTransformerFusion(nn.Module):
    def __init__(self, dim=1024, num_heads=8, mlp_ratio=4):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        q = self.q_proj(tokens)
        k = self.k_proj(tokens)
        v = self.v_proj(tokens)
        N = tokens.shape[1]
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)
        tokens = self.norm1(tokens + out)
        mlp_out = self.mlp(tokens)
        tokens = self.norm2(tokens + mlp_out)
        out_spatial = tokens.transpose(1, 2).reshape(B, C, H, W)
        return out_spatial


class EdgeAwareAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.conv_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        gx = F.conv2d(x, self.sobel_x, padding=1, groups=self.channels)
        gy = F.conv2d(x, self.sobel_y, padding=1, groups=self.channels)
        edge = torch.sqrt(gx * gx + gy * gy + 1e-6)
        feat = self.conv_branch(x)
        combined = torch.cat([edge, feat], dim=1)
        gate_map = self.gate(combined)
        return x * gate_map


class EdgeFormerNet(nn.Module):
    """Baseline 1: steps 04-06 backbone with its native sigmoid head (no evidential head)."""

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResBlock(32, 32, stride=2)
        self.enc2 = ResBlock(32, 64, stride=2)
        self.enc3 = ResBlock(64, 128, stride=2)
        self.enc4 = ResBlock(128, 256, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 1024, 3, padding=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.cstf = CrossScaleTransformerFusion(dim=1024, num_heads=8, mlp_ratio=4)
        self.up1 = nn.ConvTranspose2d(1024, 256, 2, stride=2)
        self.dec1 = ResBlock(256 + 128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResBlock(128 + 64, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ResBlock(64 + 32, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec4 = ResBlock(32 + 32, 32)
        self.eaa = EdgeAwareAttention(32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        b = self.cstf(b)
        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1, e3], dim=1))
        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d3 = self.up3(d2)
        d3 = self.dec3(torch.cat([d3, e1], dim=1))
        d4 = self.up4(d3)
        d4 = self.dec4(torch.cat([d4, e0], dim=1))
        d4 = self.eaa(d4)
        out = self.head(d4)
        return out


def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def bce_dice_loss(pred, target):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    dsc = dice_loss(pred, target)
    return bce + dsc


# ---- dataset (identical preprocessing/normalization to steps 02-03; train-only augmentation) ----
MEAN = np.array([0.44231387226534, 0.44906677331805, 0.41436488550588], dtype=np.float32)
STD = np.array([0.21254212670769, 0.19898734700646, 0.21256595675766], dtype=np.float32)

def augment_pair(img, mask):
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    angle = random.choice([0, 90, 180, 270])
    if angle != 0:
        img = img.rotate(angle)
        mask = mask.rotate(angle)
    img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
    img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))
    return img, mask

class WHUDataset(Dataset):
    def __init__(self, root, split, train=False):
        self.root = root
        self.split = split
        self.train = train
        self.files = sorted(os.listdir(os.path.join(root, split, 'Image')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.root, self.split, 'Image', fname)).convert('RGB')
        mask = Image.open(os.path.join(self.root, self.split, 'Mask', fname)).convert('L')
        if self.train:
            img, mask = augment_pair(img, mask)
        img_arr = np.asarray(img, dtype=np.float32) / 255.0
        img_arr = (img_arr - MEAN) / STD
        mask_arr = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)
        img_t = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_arr).unsqueeze(0).float()
        return img_t, mask_t

def make_loaders(root, batch_size=8, num_workers=2):
    train_ds = WHUDataset(root, 'train', train=True)
    val_ds = WHUDataset(root, 'val', train=False)
    test_ds = WHUDataset(root, 'test', train=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader


# ---- shared training loop (same discipline as step09's train_full) ----
def train_baseline(model, train_loader, val_loader, device, max_epochs, lr, seed,
                    ckpt_dir, batch_size, model_tag, compute_loss, compute_val_metric,
                    patience=2):
    set_seed(seed)
    os.makedirs(ckpt_dir, exist_ok=True)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    last_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_last.pt')
    best_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_best.pt')
    cfg_path = os.path.join(ckpt_dir, f'step10_{model_tag}_config.json')

    history = []
    best_val = float('inf')
    best_epoch = -1
    epochs_without_improvement = 0

    def save_config(reason, cur_epoch):
        config = {
            'step': '10_baselines',
            'model_tag': model_tag,
            'lr': lr,
            'batch_size': batch_size,
            'max_epochs': max_epochs,
            'epochs_run': cur_epoch,
            'python_seed': seed,
            'numpy_seed': seed,
            'torch_seed': seed,
            'optimizer': 'Adam',
            'patience': patience,
            'history': history,
            'best_epoch': best_epoch if best_epoch != -1 else None,
            'best_val_metric': best_val if best_epoch != -1 else None,
            'stopped_reason': reason,
            'last_checkpoint_path': last_ckpt_path,
            'best_checkpoint_path': best_ckpt_path,
        }
        with open(cfg_path, 'w') as f:
            json.dump(config, f, indent=2)
        return config

    stopped_reason = None
    epoch = 0
    while epoch < max_epochs:
        model.train()
        train_loss_sum = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = compute_loss(model, xb, yb)
            loss.backward()
            opt.step()
            train_loss_sum += loss.item()
            n_batches += 1
        avg_train_loss = train_loss_sum / max(n_batches, 1)

        model.eval()
        val_metric_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_metric_sum += compute_val_metric(model, xb, yb)
                n_val += 1
        avg_val_metric = val_metric_sum / max(n_val, 1)

        history.append({'epoch': epoch + 1, 'train_loss': avg_train_loss, 'val_metric': avg_val_metric})
        print(f'[{model_tag}] epoch {epoch + 1} train_loss {avg_train_loss:.5f} val_metric {avg_val_metric:.5f}')

        torch.save(model.state_dict(), last_ckpt_path)
        if avg_val_metric < best_val - 1e-4:
            best_val = avg_val_metric
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_ckpt_path)
        else:
            epochs_without_improvement += 1

        epoch += 1
        save_config('in_progress', epoch)

        if epochs_without_improvement >= patience:
            stopped_reason = f'early_stopping_patience_{patience}_at_epoch_{epoch}'
            print(f'[{model_tag}] early stopping at epoch {epoch}')
            break

    if stopped_reason is None:
        stopped_reason = f'max_epochs_reached_{max_epochs}'
    config = save_config(stopped_reason, epoch)
    print(f'[{model_tag}] stopped:', stopped_reason, 'best_epoch:', best_epoch, 'best_val:', best_val)
    return history, config

print('Part 1 class/function definitions loaded OK.')


# ===== CELL 4: build loaders =====
train_loader, val_loader, test_loader = make_loaders(root, batch_size=8, num_workers=2)
xb, yb = next(iter(train_loader))
print('sanity batch:', xb.shape, yb.shape, 'x range', xb.min().item(), xb.max().item())


# ===== CELL 5: train baseline 1 — plain sigmoid + BCE-Dice =====
model_sigmoid = EdgeFormerNet()
hist_sigmoid, cfg_sigmoid = train_baseline(
    model_sigmoid, train_loader, val_loader, device,
    max_epochs=3, lr=0.001, seed=42, ckpt_dir=ckpt_dir, batch_size=8,
    model_tag='sigmoid_baseline',
    compute_loss=lambda m, xb, yb: bce_dice_loss(m(xb), yb),
    compute_val_metric=lambda m, xb, yb: bce_dice_loss(m(xb), yb).item(),
    patience=2,
)
print(json.dumps(cfg_sigmoid, indent=2))


# ===== CELL 6: summary — copy this cell's output back to chat =====
print('=' * 70)
print('STEP 10 PART 1 SUMMARY (sigmoid baseline)')
print('=' * 70)
print('checkpoint files in', ckpt_dir, ':')
for f in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, f)
    size_str = '(' + str(round(os.path.getsize(path) / 1e6, 1)) + ' MB)' if f.endswith('.pt') else ''
    print(' ', f, size_str)
print()
print(json.dumps(cfg_sigmoid, indent=2))
print()
print('Next: Save Version -> Save & Run All (Commit) -> Output tab -> download')
print('checkpoints/ -> upload step10_sigmoid_baseline_best.pt and _last.pt into')
print('Drive Dataset WHU/checkpoints/. Then run Part 2 (temperature scaling),')
print('which needs step10_sigmoid_baseline_best.pt to exist in this same')
print('/kaggle/working/checkpoints/ folder (same session) or re-uploaded as an')
print('input dataset if starting Part 2 in a fresh session.')


CUDA available: True
Device: Tesla T4
Using dataset root: /kaggle/input/datasets/sengulgs/whu-building-dataset/WHU
train images: 5732 masks: 5732
val images: 1228 masks: 1228
test images: 1228 masks: 1228
Part 1 class/function definitions loaded OK.
sanity batch: torch.Size([8, 3, 512, 512]) torch.Size([8, 1, 512, 512]) x range -2.256760597229004 2.7686848640441895
[sigmoid_baseline] epoch 1 train_loss 0.72116 val_metric 0.60487
[sigmoid_baseline] epoch 2 train_loss 0.59840 val_metric 0.52671
[sigmoid_baseline] epoch 3 train_loss 0.54777 val_metric 0.47859
[sigmoid_baseline] stopped: max_epochs_reached_3 best_epoch: 3 best_val: 0.47859334036127316
{
  "step": "10_baselines",
  "model_tag": "sigmoid_baseline",
  "lr": 0.001,
  "batch_size": 8,
  "max_epochs": 3,
  "epochs_run": 3,
  "python_seed": 42,
  "numpy_seed": 42,
  "torch_seed": 42,
  "optimizer": "Adam",
  "patience": 2,
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.7211602699406986,
      "val_metric": 0.6048735

In [3]:
# ============================================================================
# CalEdgeFormer — Step 10, Part 2 of 4: temperature scaling (self-contained)
# ============================================================================
# Purpose: post-hoc calibration on the sigmoid baseline from Part 1. This is
# NOT a training run — it loads Part 1's checkpoint, runs it once over val/
# to get logits, then fits a single scalar temperature T (Guo et al. 2017).
# Fast (a couple of minutes), no epochs.
#
# REQUIRES: step10_sigmoid_baseline_best.pt from Part 1.
#   - If you're running this in the SAME Kaggle session as Part 1, it's
#     already sitting in /kaggle/working/checkpoints/ — nothing to do.
#   - If you're running this in a FRESH session, add the checkpoint as an
#     input dataset first (Add Data -> Upload -> the .pt file you downloaded
#     after Part 1), then this script will find it under /kaggle/input.
#
# Paste each "CELL" block into its own Kaggle notebook cell, in order.
# Settings -> Accelerator -> GPU T4 x2 or GPU P100 (a GPU still helps for the
# one forward pass over val/, though this part is far lighter than training).
# ============================================================================


# ===== CELL 1: environment check =====
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ===== CELL 2: locate and verify the WHU dataset under /kaggle/input =====
import os

def find_whu_root(base='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, s)) for s in ('train', 'val', 'test')):
            if all(os.path.isdir(os.path.join(dirpath, s, 'Image')) and
                   os.path.isdir(os.path.join(dirpath, s, 'Mask'))
                   for s in ('train', 'val', 'test')):
                return dirpath
    return None

root = find_whu_root()
if root is None:
    # root = '/kaggle/input/whu-building-dataset/WHU'  # set manually if needed
    raise RuntimeError('Could not auto-detect the WHU dataset under /kaggle/input.')

print('Using dataset root:', root)
val_count = len(os.listdir(os.path.join(root, 'val', 'Image')))
print('val images:', val_count)

ckpt_dir = '/kaggle/working/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)


# ===== CELL 3: locate Part 1's sigmoid checkpoint =====
SIGMOID_CKPT_NAME = 'step10_sigmoid_baseline_best.pt'

def find_file(filename, search_roots=('/kaggle/working', '/kaggle/input')):
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None

sigmoid_ckpt_path = find_file(SIGMOID_CKPT_NAME)
if sigmoid_ckpt_path is None:
    raise RuntimeError(
        f'Could not find {SIGMOID_CKPT_NAME} under /kaggle/working or /kaggle/input. '
        'If this is a fresh session (not the same one Part 1 ran in), add it as an '
        'input dataset first (Add Data -> Upload -> the .pt file you downloaded '
        'after Part 1), then re-run this cell.'
    )
print('Found sigmoid checkpoint at:', sigmoid_ckpt_path)


# ===== CELL 4: class/function definitions needed to load the sigmoid model =====
import json
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class CrossScaleTransformerFusion(nn.Module):
    def __init__(self, dim=1024, num_heads=8, mlp_ratio=4):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        q = self.q_proj(tokens)
        k = self.k_proj(tokens)
        v = self.v_proj(tokens)
        N = tokens.shape[1]
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)
        tokens = self.norm1(tokens + out)
        mlp_out = self.mlp(tokens)
        tokens = self.norm2(tokens + mlp_out)
        out_spatial = tokens.transpose(1, 2).reshape(B, C, H, W)
        return out_spatial


class EdgeAwareAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.conv_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        gx = F.conv2d(x, self.sobel_x, padding=1, groups=self.channels)
        gy = F.conv2d(x, self.sobel_y, padding=1, groups=self.channels)
        edge = torch.sqrt(gx * gx + gy * gy + 1e-6)
        feat = self.conv_branch(x)
        combined = torch.cat([edge, feat], dim=1)
        gate_map = self.gate(combined)
        return x * gate_map


class EdgeFormerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResBlock(32, 32, stride=2)
        self.enc2 = ResBlock(32, 64, stride=2)
        self.enc3 = ResBlock(64, 128, stride=2)
        self.enc4 = ResBlock(128, 256, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 1024, 3, padding=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.cstf = CrossScaleTransformerFusion(dim=1024, num_heads=8, mlp_ratio=4)
        self.up1 = nn.ConvTranspose2d(1024, 256, 2, stride=2)
        self.dec1 = ResBlock(256 + 128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResBlock(128 + 64, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ResBlock(64 + 32, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec4 = ResBlock(32 + 32, 32)
        self.eaa = EdgeAwareAttention(32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        b = self.cstf(b)
        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1, e3], dim=1))
        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d3 = self.up3(d2)
        d3 = self.dec3(torch.cat([d3, e1], dim=1))
        d4 = self.up4(d3)
        d4 = self.dec4(torch.cat([d4, e0], dim=1))
        d4 = self.eaa(d4)
        out = self.head(d4)
        return out


# ---- dataset (val/ only needed here — no augmentation regardless) ----
MEAN = np.array([0.44231387226534, 0.44906677331805, 0.41436488550588], dtype=np.float32)
STD = np.array([0.21254212670769, 0.19898734700646, 0.21256595675766], dtype=np.float32)

class WHUDataset(Dataset):
    def __init__(self, root, split, train=False):
        self.root = root
        self.split = split
        self.train = train
        self.files = sorted(os.listdir(os.path.join(root, split, 'Image')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.root, self.split, 'Image', fname)).convert('RGB')
        mask = Image.open(os.path.join(self.root, self.split, 'Mask', fname)).convert('L')
        img_arr = np.asarray(img, dtype=np.float32) / 255.0
        img_arr = (img_arr - MEAN) / STD
        mask_arr = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)
        img_t = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_arr).unsqueeze(0).float()
        return img_t, mask_t


def fit_temperature_scaling(model, val_loader, device, ckpt_dir, sigmoid_ckpt_path):
    model.eval()
    logits_list = []
    labels_list = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            logits_list.append(logits.cpu())
            labels_list.append(yb)
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list)

    T = torch.nn.Parameter(torch.ones(1) * 1.5)
    optimizer = torch.optim.LBFGS([T], lr=0.05, max_iter=50)

    def closure():
        optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(logits / T.clamp(min=0.05), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    T_final = float(T.clamp(min=0.05).item())

    pre_nll = F.binary_cross_entropy_with_logits(logits, labels).item()
    post_nll = F.binary_cross_entropy_with_logits(logits / T_final, labels).item()

    os.makedirs(ckpt_dir, exist_ok=True)
    cfg_path = os.path.join(ckpt_dir, 'step10_temp_scaling_config.json')
    config = {
        'step': '10_baselines',
        'model_tag': 'temp_scaling',
        'base_model_checkpoint': sigmoid_ckpt_path,
        'method': 'single scalar temperature T fit via LBFGS minimizing NLL on val/ logits (Guo et al. 2017)',
        'temperature': T_final,
        'val_nll_before_scaling': pre_nll,
        'val_nll_after_scaling': post_nll,
    }
    with open(cfg_path, 'w') as f:
        json.dump(config, f, indent=2)
    print('temperature scaling: T=', T_final, 'val_nll before=', pre_nll, 'after=', post_nll)
    return T_final, config

print('Part 2 class/function definitions loaded OK.')


# ===== CELL 5: build val loader + load the sigmoid checkpoint =====
val_ds = WHUDataset(root, 'val', train=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
print('val batches:', len(val_loader))

model_for_ts = EdgeFormerNet().to(device)
model_for_ts.load_state_dict(torch.load(sigmoid_ckpt_path, map_location=device))
print('Loaded sigmoid checkpoint from', sigmoid_ckpt_path)


# ===== CELL 6: fit temperature scaling =====
T_final, cfg_temp = fit_temperature_scaling(model_for_ts, val_loader, device, ckpt_dir, sigmoid_ckpt_path)
print(json.dumps(cfg_temp, indent=2))


# ===== CELL 7: summary — copy this cell's output back to chat =====
print('=' * 70)
print('STEP 10 PART 2 SUMMARY (temperature scaling)')
print('=' * 70)
print('checkpoint dir contents (', ckpt_dir, '):')
for f in sorted(os.listdir(ckpt_dir)):
    print(' ', f)
print()
print(json.dumps(cfg_temp, indent=2))
print()
print('No .pt file for this part — temperature scaling only produces a config')
print('JSON (the scalar T value + a pointer to the sigmoid checkpoint it was')
print('fit on). Nothing new to download here beyond step10_temp_scaling_config.json')
print('if you want it as a standalone file; it is also already embedded above.')


CUDA available: True
Device: Tesla T4
Using dataset root: /kaggle/input/datasets/sengulgs/whu-building-dataset/WHU
val images: 1228
Found sigmoid checkpoint at: /kaggle/working/checkpoints/step10_sigmoid_baseline_best.pt
Part 2 class/function definitions loaded OK.
val batches: 154
Loaded sigmoid checkpoint from /kaggle/working/checkpoints/step10_sigmoid_baseline_best.pt
temperature scaling: T= 1.312425136566162 val_nll before= 0.1209295317530632 after= 0.11493748426437378
{
  "step": "10_baselines",
  "model_tag": "temp_scaling",
  "base_model_checkpoint": "/kaggle/working/checkpoints/step10_sigmoid_baseline_best.pt",
  "method": "single scalar temperature T fit via LBFGS minimizing NLL on val/ logits (Guo et al. 2017)",
  "temperature": 1.312425136566162,
  "val_nll_before_scaling": 0.1209295317530632,
  "val_nll_after_scaling": 0.11493748426437378
}
STEP 10 PART 2 SUMMARY (temperature scaling)
checkpoint dir contents ( /kaggle/working/checkpoints ):
  step10_sigmoid_baseline_best.pt

In [4]:
# ============================================================================
# CalEdgeFormer — Step 10, Part 3 of 4: MC-Dropout (self-contained)
# ============================================================================
# Purpose: step 10 already PASSED (config JSON already recorded) — this run
# exists only to regenerate the real .pt weight file, since the original
# Kaggle session's checkpoints folder was lost before it could be downloaded.
#
# Paste each "CELL" block into its own Kaggle notebook cell, in order.
# Settings -> Accelerator -> GPU T4 x2 or GPU P100.
#
# This part is fully self-contained and INDEPENDENT of Parts 1/2/4 — no
# checkpoint from any other part is needed. Run it any time.
#
# After it finishes: Save Version -> Save & Run All (Commit) -> Output tab ->
# download checkpoints/ -> upload step10_mcdropout_best.pt and _last.pt into
# Drive's Dataset WHU/checkpoints/.
# ============================================================================


# ===== CELL 1: environment check =====
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (stop and enable GPU accelerator!)')
assert torch.cuda.is_available(), 'No GPU — enable it in Settings before continuing.'
device = torch.device('cuda')


# ===== CELL 2: locate and verify the WHU dataset under /kaggle/input =====
import os

def find_whu_root(base='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, s)) for s in ('train', 'val', 'test')):
            if all(os.path.isdir(os.path.join(dirpath, s, 'Image')) and
                   os.path.isdir(os.path.join(dirpath, s, 'Mask'))
                   for s in ('train', 'val', 'test')):
                return dirpath
    return None

root = find_whu_root()
if root is None:
    # root = '/kaggle/input/whu-building-dataset/WHU'  # set manually if needed
    raise RuntimeError('Could not auto-detect the WHU dataset under /kaggle/input.')

print('Using dataset root:', root)
for s in ['train', 'val', 'test']:
    imgs = os.listdir(os.path.join(root, s, 'Image'))
    msks = os.listdir(os.path.join(root, s, 'Mask'))
    print(s, 'images:', len(imgs), 'masks:', len(msks))

ckpt_dir = '/kaggle/working/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)


# ===== CELL 3: class/function definitions needed for MC-Dropout =====
import json
import math
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class CrossScaleTransformerFusion(nn.Module):
    def __init__(self, dim=1024, num_heads=8, mlp_ratio=4):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        q = self.q_proj(tokens)
        k = self.k_proj(tokens)
        v = self.v_proj(tokens)
        N = tokens.shape[1]
        q = q.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)
        tokens = self.norm1(tokens + out)
        mlp_out = self.mlp(tokens)
        tokens = self.norm2(tokens + mlp_out)
        out_spatial = tokens.transpose(1, 2).reshape(B, C, H, W)
        return out_spatial


class EdgeAwareAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1))
        self.conv_branch = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        gx = F.conv2d(x, self.sobel_x, padding=1, groups=self.channels)
        gy = F.conv2d(x, self.sobel_y, padding=1, groups=self.channels)
        edge = torch.sqrt(gx * gx + gy * gy + 1e-6)
        feat = self.conv_branch(x)
        combined = torch.cat([edge, feat], dim=1)
        gate_map = self.gate(combined)
        return x * gate_map


class EdgeFormerNetMCDropout(nn.Module):
    """Same backbone as the steps 04-06 model, plus Dropout2d after the
    bottleneck/CSTF and after each decoder block, kept active at inference
    (MC Dropout, Gal & Ghahramani 2016) to sample an approximate posterior."""

    def __init__(self, p=0.2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResBlock(32, 32, stride=2)
        self.enc2 = ResBlock(32, 64, stride=2)
        self.enc3 = ResBlock(64, 128, stride=2)
        self.enc4 = ResBlock(128, 256, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 1024, 3, padding=1, bias=False),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
        )
        self.cstf = CrossScaleTransformerFusion(dim=1024, num_heads=8, mlp_ratio=4)
        self.drop_bottleneck = nn.Dropout2d(p)
        self.up1 = nn.ConvTranspose2d(1024, 256, 2, stride=2)
        self.dec1 = ResBlock(256 + 128, 256)
        self.drop1 = nn.Dropout2d(p)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResBlock(128 + 64, 128)
        self.drop2 = nn.Dropout2d(p)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ResBlock(64 + 32, 64)
        self.drop3 = nn.Dropout2d(p)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec4 = ResBlock(32 + 32, 32)
        self.eaa = EdgeAwareAttention(32)
        self.head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        b = self.cstf(b)
        b = self.drop_bottleneck(b)
        d1 = self.up1(b)
        d1 = self.drop1(self.dec1(torch.cat([d1, e3], dim=1)))
        d2 = self.up2(d1)
        d2 = self.drop2(self.dec2(torch.cat([d2, e2], dim=1)))
        d3 = self.up3(d2)
        d3 = self.drop3(self.dec3(torch.cat([d3, e1], dim=1)))
        d4 = self.up4(d3)
        d4 = self.dec4(torch.cat([d4, e0], dim=1))
        d4 = self.eaa(d4)
        return self.head(d4)


def enable_mc_dropout(model):
    """Set only Dropout layers to train mode, everything else (esp. BatchNorm) stays eval."""
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()


@torch.no_grad()
def mc_dropout_predict(model, xb, n_samples=25):
    enable_mc_dropout(model)
    probs = torch.stack([torch.sigmoid(model(xb)) for _ in range(n_samples)], dim=0)
    mean_p = probs.mean(dim=0)
    var_p = probs.var(dim=0)
    return mean_p, var_p


def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def bce_dice_loss(pred, target):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    dsc = dice_loss(pred, target)
    return bce + dsc


# ---- dataset (identical preprocessing/normalization to steps 02-03; train-only augmentation) ----
MEAN = np.array([0.44231387226534, 0.44906677331805, 0.41436488550588], dtype=np.float32)
STD = np.array([0.21254212670769, 0.19898734700646, 0.21256595675766], dtype=np.float32)

def augment_pair(img, mask):
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    angle = random.choice([0, 90, 180, 270])
    if angle != 0:
        img = img.rotate(angle)
        mask = mask.rotate(angle)
    img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
    img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))
    return img, mask

class WHUDataset(Dataset):
    def __init__(self, root, split, train=False):
        self.root = root
        self.split = split
        self.train = train
        self.files = sorted(os.listdir(os.path.join(root, split, 'Image')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.root, self.split, 'Image', fname)).convert('RGB')
        mask = Image.open(os.path.join(self.root, self.split, 'Mask', fname)).convert('L')
        if self.train:
            img, mask = augment_pair(img, mask)
        img_arr = np.asarray(img, dtype=np.float32) / 255.0
        img_arr = (img_arr - MEAN) / STD
        mask_arr = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)
        img_t = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_arr).unsqueeze(0).float()
        return img_t, mask_t

def make_loaders(root, batch_size=8, num_workers=2):
    train_ds = WHUDataset(root, 'train', train=True)
    val_ds = WHUDataset(root, 'val', train=False)
    test_ds = WHUDataset(root, 'test', train=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader


# ---- shared training loop (same discipline as step09's train_full) ----
def train_baseline(model, train_loader, val_loader, device, max_epochs, lr, seed,
                    ckpt_dir, batch_size, model_tag, compute_loss, compute_val_metric,
                    patience=2):
    set_seed(seed)
    os.makedirs(ckpt_dir, exist_ok=True)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    last_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_last.pt')
    best_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_best.pt')
    cfg_path = os.path.join(ckpt_dir, f'step10_{model_tag}_config.json')

    history = []
    best_val = float('inf')
    best_epoch = -1
    epochs_without_improvement = 0

    def save_config(reason, cur_epoch):
        config = {
            'step': '10_baselines',
            'model_tag': model_tag,
            'lr': lr,
            'batch_size': batch_size,
            'max_epochs': max_epochs,
            'epochs_run': cur_epoch,
            'python_seed': seed,
            'numpy_seed': seed,
            'torch_seed': seed,
            'optimizer': 'Adam',
            'patience': patience,
            'history': history,
            'best_epoch': best_epoch if best_epoch != -1 else None,
            'best_val_metric': best_val if best_epoch != -1 else None,
            'stopped_reason': reason,
            'last_checkpoint_path': last_ckpt_path,
            'best_checkpoint_path': best_ckpt_path,
        }
        with open(cfg_path, 'w') as f:
            json.dump(config, f, indent=2)
        return config

    stopped_reason = None
    epoch = 0
    while epoch < max_epochs:
        model.train()
        train_loss_sum = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = compute_loss(model, xb, yb)
            loss.backward()
            opt.step()
            train_loss_sum += loss.item()
            n_batches += 1
        avg_train_loss = train_loss_sum / max(n_batches, 1)

        model.eval()
        val_metric_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_metric_sum += compute_val_metric(model, xb, yb)
                n_val += 1
        avg_val_metric = val_metric_sum / max(n_val, 1)

        history.append({'epoch': epoch + 1, 'train_loss': avg_train_loss, 'val_metric': avg_val_metric})
        print(f'[{model_tag}] epoch {epoch + 1} train_loss {avg_train_loss:.5f} val_metric {avg_val_metric:.5f}')

        torch.save(model.state_dict(), last_ckpt_path)
        if avg_val_metric < best_val - 1e-4:
            best_val = avg_val_metric
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_ckpt_path)
        else:
            epochs_without_improvement += 1

        epoch += 1
        save_config('in_progress', epoch)

        if epochs_without_improvement >= patience:
            stopped_reason = f'early_stopping_patience_{patience}_at_epoch_{epoch}'
            print(f'[{model_tag}] early stopping at epoch {epoch}')
            break

    if stopped_reason is None:
        stopped_reason = f'max_epochs_reached_{max_epochs}'
    config = save_config(stopped_reason, epoch)
    print(f'[{model_tag}] stopped:', stopped_reason, 'best_epoch:', best_epoch, 'best_val:', best_val)
    return history, config

print('Part 3 class/function definitions loaded OK.')


# ===== CELL 4: build loaders =====
train_loader, val_loader, test_loader = make_loaders(root, batch_size=8, num_workers=2)
xb, yb = next(iter(train_loader))
print('sanity batch:', xb.shape, yb.shape, 'x range', xb.min().item(), xb.max().item())


# ===== CELL 5: train MC-Dropout =====
model_mcdrop = EdgeFormerNetMCDropout(p=0.2)
hist_mcdrop, cfg_mcdrop = train_baseline(
    model_mcdrop, train_loader, val_loader, device,
    max_epochs=3, lr=0.001, seed=42, ckpt_dir=ckpt_dir, batch_size=8,
    model_tag='mcdropout',
    compute_loss=lambda m, xb, yb: bce_dice_loss(m(xb), yb),
    compute_val_metric=lambda m, xb, yb: bce_dice_loss(m(xb), yb).item(),
    patience=2,
)
print(json.dumps(cfg_mcdrop, indent=2))


# ===== CELL 6: MC-Dropout sanity check (confirms sampling actually varies) =====
mcdrop_best = EdgeFormerNetMCDropout(p=0.2).to(device)
mcdrop_best.load_state_dict(torch.load(cfg_mcdrop['best_checkpoint_path'], map_location=device))
xb_s, yb_s = next(iter(val_loader))
mean_p, var_p = mc_dropout_predict(mcdrop_best, xb_s.to(device), n_samples=25)
print('MC-Dropout sanity: mean_p range', mean_p.min().item(), mean_p.max().item(),
      'var_p range', var_p.min().item(), var_p.max().item())


# ===== CELL 7: summary — copy this cell's output back to chat =====
print('=' * 70)
print('STEP 10 PART 3 SUMMARY (MC-Dropout)')
print('=' * 70)
print('checkpoint files in', ckpt_dir, ':')
for f in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, f)
    size_str = '(' + str(round(os.path.getsize(path) / 1e6, 1)) + ' MB)' if f.endswith('.pt') else ''
    print(' ', f, size_str)
print()
print(json.dumps(cfg_mcdrop, indent=2))
print()
print('Next: Save Version -> Save & Run All (Commit) -> Output tab -> download')
print('checkpoints/ -> upload step10_mcdropout_best.pt and _last.pt into')
print('Drive Dataset WHU/checkpoints/. Then run Part 4 (UANet) — also fully')
print('independent, no checkpoint dependency.')


CUDA available: True
Device: Tesla T4
Using dataset root: /kaggle/input/datasets/sengulgs/whu-building-dataset/WHU
train images: 5732 masks: 5732
val images: 1228 masks: 1228
test images: 1228 masks: 1228
Part 3 class/function definitions loaded OK.
sanity batch: torch.Size([8, 3, 512, 512]) torch.Size([8, 1, 512, 512]) x range -2.256760597229004 2.7686848640441895
[mcdropout] epoch 1 train_loss 0.71406 val_metric 0.52597
[mcdropout] epoch 2 train_loss 0.54522 val_metric 0.47286
[mcdropout] epoch 3 train_loss 0.49431 val_metric 0.43842
[mcdropout] stopped: max_epochs_reached_3 best_epoch: 3 best_val: 0.4384193199795562
{
  "step": "10_baselines",
  "model_tag": "mcdropout",
  "lr": 0.001,
  "batch_size": 8,
  "max_epochs": 3,
  "epochs_run": 3,
  "python_seed": 42,
  "numpy_seed": 42,
  "torch_seed": 42,
  "optimizer": "Adam",
  "patience": 2,
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.7140639927134168,
      "val_metric": 0.5259728294301342
    },
    {
      "epoch"

In [5]:
# ============================================================================
# CalEdgeFormer — Step 10, Part 4 of 4: UANet reproduction (self-contained)
# ============================================================================
# Purpose: step 10 already PASSED (config JSON already recorded) — this run
# exists only to regenerate the real .pt weight file, since the original
# Kaggle session's checkpoints folder was lost before it could be downloaded.
#
# Paste each "CELL" block into its own Kaggle notebook cell, in order.
# Settings -> Accelerator -> GPU T4 x2 or GPU P100.
#
# This part is fully self-contained and INDEPENDENT of Parts 1/2/3 — no
# checkpoint from any other part is needed. This is the last of the 4 parts.
#
# After it finishes: Save Version -> Save & Run All (Commit) -> Output tab ->
# download checkpoints/ -> upload step10_uanet_best.pt and _last.pt into
# Drive's Dataset WHU/checkpoints/. At that point ALL of step 10's real
# weight files (sigmoid, mcdropout, uanet — temp_scaling has no .pt of its
# own) will be recovered and safely in Drive alongside steps 04-09.
# ============================================================================


# ===== CELL 1: environment check =====
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (stop and enable GPU accelerator!)')
assert torch.cuda.is_available(), 'No GPU — enable it in Settings before continuing.'
device = torch.device('cuda')


# ===== CELL 2: locate and verify the WHU dataset under /kaggle/input =====
import os

def find_whu_root(base='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(base):
        if all(os.path.isdir(os.path.join(dirpath, s)) for s in ('train', 'val', 'test')):
            if all(os.path.isdir(os.path.join(dirpath, s, 'Image')) and
                   os.path.isdir(os.path.join(dirpath, s, 'Mask'))
                   for s in ('train', 'val', 'test')):
                return dirpath
    return None

root = find_whu_root()
if root is None:
    # root = '/kaggle/input/whu-building-dataset/WHU'  # set manually if needed
    raise RuntimeError('Could not auto-detect the WHU dataset under /kaggle/input.')

print('Using dataset root:', root)
for s in ['train', 'val', 'test']:
    imgs = os.listdir(os.path.join(root, s, 'Image'))
    msks = os.listdir(os.path.join(root, s, 'Mask'))
    print(s, 'images:', len(imgs), 'masks:', len(msks))

ckpt_dir = '/kaggle/working/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)


# ===== CELL 3: class/function definitions needed for UANet =====
import json
import math
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class UANet(nn.Module):
    """Reproduction of an uncertainty-aware building-extraction network (UANet-style):
    a residual encoder-decoder with two output heads (mean logit + log-variance),
    trained with a heteroscedastic aleatoric-uncertainty loss (Kendall & Gal, 2017).
    Reproduced from the general description in the literature — no official released
    code exists to reference directly."""

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.enc1 = ResBlock(32, 32, stride=2)
        self.enc2 = ResBlock(32, 64, stride=2)
        self.enc3 = ResBlock(64, 128, stride=2)
        self.enc4 = ResBlock(128, 256, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec1 = ResBlock(256 + 128, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResBlock(128 + 64, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ResBlock(64 + 32, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec4 = ResBlock(32 + 32, 32)
        self.mean_head = nn.Conv2d(32, 1, 1)
        self.logvar_head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1, e3], dim=1))
        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d3 = self.up3(d2)
        d3 = self.dec3(torch.cat([d3, e1], dim=1))
        d4 = self.up4(d3)
        d4 = self.dec4(torch.cat([d4, e0], dim=1))
        mean_logit = self.mean_head(d4)
        log_var = self.logvar_head(d4).clamp(-6, 6)
        return {'mean_logit': mean_logit, 'log_var': log_var}


def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def uanet_loss(out, target, n_mc=8, dice_weight=1.0):
    mean_logit = out['mean_logit']
    log_var = out['log_var']
    std = torch.exp(0.5 * log_var)
    bce_total = 0.0
    for _ in range(n_mc):
        eps = torch.randn_like(mean_logit)
        distorted = mean_logit + std * eps
        bce_total = bce_total + F.binary_cross_entropy_with_logits(distorted, target)
    bce = bce_total / n_mc
    dsc = dice_loss(mean_logit, target)
    return bce + dice_weight * dsc, bce, dsc


# ---- dataset (identical preprocessing/normalization to steps 02-03; train-only augmentation) ----
MEAN = np.array([0.44231387226534, 0.44906677331805, 0.41436488550588], dtype=np.float32)
STD = np.array([0.21254212670769, 0.19898734700646, 0.21256595675766], dtype=np.float32)

def augment_pair(img, mask):
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    angle = random.choice([0, 90, 180, 270])
    if angle != 0:
        img = img.rotate(angle)
        mask = mask.rotate(angle)
    img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
    img = TF.adjust_contrast(img, random.uniform(0.8, 1.2))
    return img, mask

class WHUDataset(Dataset):
    def __init__(self, root, split, train=False):
        self.root = root
        self.split = split
        self.train = train
        self.files = sorted(os.listdir(os.path.join(root, split, 'Image')))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.root, self.split, 'Image', fname)).convert('RGB')
        mask = Image.open(os.path.join(self.root, self.split, 'Mask', fname)).convert('L')
        if self.train:
            img, mask = augment_pair(img, mask)
        img_arr = np.asarray(img, dtype=np.float32) / 255.0
        img_arr = (img_arr - MEAN) / STD
        mask_arr = (np.asarray(mask, dtype=np.float32) > 127.0).astype(np.float32)
        img_t = torch.from_numpy(img_arr.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_arr).unsqueeze(0).float()
        return img_t, mask_t

def make_loaders(root, batch_size=8, num_workers=2):
    train_ds = WHUDataset(root, 'train', train=True)
    val_ds = WHUDataset(root, 'val', train=False)
    test_ds = WHUDataset(root, 'test', train=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader


# ---- shared training loop (same discipline as step09's train_full) ----
def train_baseline(model, train_loader, val_loader, device, max_epochs, lr, seed,
                    ckpt_dir, batch_size, model_tag, compute_loss, compute_val_metric,
                    patience=2):
    set_seed(seed)
    os.makedirs(ckpt_dir, exist_ok=True)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    last_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_last.pt')
    best_ckpt_path = os.path.join(ckpt_dir, f'step10_{model_tag}_best.pt')
    cfg_path = os.path.join(ckpt_dir, f'step10_{model_tag}_config.json')

    history = []
    best_val = float('inf')
    best_epoch = -1
    epochs_without_improvement = 0

    def save_config(reason, cur_epoch):
        config = {
            'step': '10_baselines',
            'model_tag': model_tag,
            'lr': lr,
            'batch_size': batch_size,
            'max_epochs': max_epochs,
            'epochs_run': cur_epoch,
            'python_seed': seed,
            'numpy_seed': seed,
            'torch_seed': seed,
            'optimizer': 'Adam',
            'patience': patience,
            'history': history,
            'best_epoch': best_epoch if best_epoch != -1 else None,
            'best_val_metric': best_val if best_epoch != -1 else None,
            'stopped_reason': reason,
            'last_checkpoint_path': last_ckpt_path,
            'best_checkpoint_path': best_ckpt_path,
        }
        with open(cfg_path, 'w') as f:
            json.dump(config, f, indent=2)
        return config

    stopped_reason = None
    epoch = 0
    while epoch < max_epochs:
        model.train()
        train_loss_sum = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = compute_loss(model, xb, yb)
            loss.backward()
            opt.step()
            train_loss_sum += loss.item()
            n_batches += 1
        avg_train_loss = train_loss_sum / max(n_batches, 1)

        model.eval()
        val_metric_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_metric_sum += compute_val_metric(model, xb, yb)
                n_val += 1
        avg_val_metric = val_metric_sum / max(n_val, 1)

        history.append({'epoch': epoch + 1, 'train_loss': avg_train_loss, 'val_metric': avg_val_metric})
        print(f'[{model_tag}] epoch {epoch + 1} train_loss {avg_train_loss:.5f} val_metric {avg_val_metric:.5f}')

        torch.save(model.state_dict(), last_ckpt_path)
        if avg_val_metric < best_val - 1e-4:
            best_val = avg_val_metric
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_ckpt_path)
        else:
            epochs_without_improvement += 1

        epoch += 1
        save_config('in_progress', epoch)

        if epochs_without_improvement >= patience:
            stopped_reason = f'early_stopping_patience_{patience}_at_epoch_{epoch}'
            print(f'[{model_tag}] early stopping at epoch {epoch}')
            break

    if stopped_reason is None:
        stopped_reason = f'max_epochs_reached_{max_epochs}'
    config = save_config(stopped_reason, epoch)
    print(f'[{model_tag}] stopped:', stopped_reason, 'best_epoch:', best_epoch, 'best_val:', best_val)
    return history, config

print('Part 4 class/function definitions loaded OK.')


# ===== CELL 4: build loaders =====
train_loader, val_loader, test_loader = make_loaders(root, batch_size=8, num_workers=2)
xb, yb = next(iter(train_loader))
print('sanity batch:', xb.shape, yb.shape, 'x range', xb.min().item(), xb.max().item())


# ===== CELL 5: train UANet =====
model_uanet = UANet()
hist_uanet, cfg_uanet = train_baseline(
    model_uanet, train_loader, val_loader, device,
    max_epochs=3, lr=0.001, seed=42, ckpt_dir=ckpt_dir, batch_size=8,
    model_tag='uanet',
    compute_loss=lambda m, xb, yb: uanet_loss(m(xb), yb)[0],
    compute_val_metric=lambda m, xb, yb: uanet_loss(m(xb), yb)[0].item(),
    patience=2,
)
print(json.dumps(cfg_uanet, indent=2))


# ===== CELL 6: summary — copy this cell's output back to chat =====
print('=' * 70)
print('STEP 10 PART 4 SUMMARY (UANet) — LAST OF 4 PARTS')
print('=' * 70)
print('checkpoint files in', ckpt_dir, ':')
for f in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, f)
    size_str = '(' + str(round(os.path.getsize(path) / 1e6, 1)) + ' MB)' if f.endswith('.pt') else ''
    print(' ', f, size_str)
print()
print(json.dumps(cfg_uanet, indent=2))
print()
print('Next: Save Version -> Save & Run All (Commit) -> Output tab -> download')
print('checkpoints/ -> upload step10_uanet_best.pt and _last.pt into Drive')
print('Dataset WHU/checkpoints/. Once this is done, all of step 10\'s real')
print('weight files are recovered (sigmoid, mcdropout, uanet .pt files, plus')
print('temp_scaling\'s config-only artifact) — step 10 is then fully closed')
print('out, not just pass-condition-met but with usable checkpoints too.')


CUDA available: True
Device: Tesla T4
Using dataset root: /kaggle/input/datasets/sengulgs/whu-building-dataset/WHU
train images: 5732 masks: 5732
val images: 1228 masks: 1228
test images: 1228 masks: 1228
Part 4 class/function definitions loaded OK.
sanity batch: torch.Size([8, 3, 512, 512]) torch.Size([8, 1, 512, 512]) x range -2.256760597229004 2.7686848640441895
[uanet] epoch 1 train_loss 0.63589 val_metric 0.50717
[uanet] epoch 2 train_loss 0.47359 val_metric 0.48058
[uanet] epoch 3 train_loss 0.42643 val_metric 0.40823
[uanet] stopped: max_epochs_reached_3 best_epoch: 3 best_val: 0.4082338034913137
{
  "step": "10_baselines",
  "model_tag": "uanet",
  "lr": 0.001,
  "batch_size": 8,
  "max_epochs": 3,
  "epochs_run": 3,
  "python_seed": 42,
  "numpy_seed": 42,
  "torch_seed": 42,
  "optimizer": "Adam",
  "patience": 2,
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.6358932613826996,
      "val_metric": 0.5071741425758832
    },
    {
      "epoch": 2,
      "train_lo